# 계산수학 01. Monte Carlo 방법

## 학습 목표
- 핵심 정의와 정리를 자신의 말로 설명한다.
- 기본 개념 예제를 손계산으로 확인한다.
- Python 코드와 시각화를 통해 직관을 검산한다.

## 핵심 개념
- Monte Carlo
- 계산실험
- 수렴성
- 복잡도

## 기본 개념 예제
무작위 표본으로 면적과 기대값을 추정한다.

## 실제 응용 예제
고차원 적분을 확률적 샘플링으로 근사한다.

## 이론 정리

### 정의와 관점
- 계산수학은 수학적 모델, 알고리즘, 구현, 검증을 하나의 실험 체계로 묶는다.
- Monte Carlo 방법은 난수 표본으로 적분, 기대값, 확률을 근사한다.
- 재현성은 같은 입력, 같은 코드, 같은 환경에서 같은 결과를 다시 얻을 수 있음을 뜻한다.
- 계산복잡도는 문제 크기가 커질 때 시간과 메모리가 어떻게 증가하는지 설명한다.

### 핵심 명제와 정리
- Monte Carlo 오차는 많은 경우 표본 수의 제곱근에 반비례해 줄어든다.
- 벡터화와 희소구조 활용은 같은 수학을 훨씬 빠르게 계산하게 한다.
- 수치 실험은 해석해, 보존량, 수렴률, 민감도 분석으로 검증해야 한다.
- 난수 seed와 환경 기록은 결과 재현의 최소 조건이다.

### 계산과 학습 절차
- 문제 정의, 입력, 알고리즘, 출력, 평가 지표를 먼저 분리한다.
- 작은 장난감 예제로 구현을 검산한 뒤 큰 문제로 확장한다.
- 실험 결과는 표, 그림, 로그, 파라미터 설정을 함께 저장한다.
- 무작위 실험은 seed를 고정한 결과와 여러 seed 평균을 모두 확인한다.

### 자주 생기는 오해
- 한 번의 시뮬레이션 결과만 보고 결론을 내리면 표본 변동성을 놓친다.
- 빠른 코드는 정확한 코드와 다르며, 최적화 전에 검증이 먼저다.
- 출력 그림만 저장하고 입력 파라미터를 잃으면 재현이 불가능해진다.

### 증명으로 연결하기
- 계산 실험의 신뢰성은 수학적 오차 추정, 구현 테스트, 독립 재현으로 보강한다.
- Monte Carlo 분석은 표본평균의 불편성, 분산, 큰수의 법칙을 사용한다.
- 알고리즘 설명은 의사코드, 불변식, 종료조건, 복잡도 분석을 포함해야 한다.

### 이 챕터에서 꼭 확인할 질문
- 기본 개념 예제 "무작위 표본으로 면적과 기대값을 추정한다."에서 실제로 사용한 정의는 무엇인가?
- 응용 예제 "고차원 적분을 확률적 샘플링으로 근사한다."에서 어떤 가정이 현실을 단순화하고 있는가?
- 코드가 연속 대상을 이산화한다면, 격자나 표본 수를 바꾸어도 결론이 유지되는가?
- 손계산 가능한 작은 사례와 노트북 결과가 같은 결론을 주는가?

## 0. 실행 준비

아래 셀은 프로젝트 루트의 `common/math_viz.py`를 찾아서 현재 챕터의 출력 폴더를 자동으로 설정합니다. Jupyter Lab을 프로젝트 루트에서 열면 가장 안정적으로 동작합니다.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display


CHAPTER_RELATIVE_DIR = Path("4학년_심화_과목과_연구_주제/12_계산수학/ch01_Monte_Carlo_방법")


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "common" / "math_viz.py").exists():
            return candidate
    raise RuntimeError("common/math_viz.py를 찾지 못했습니다. Jupyter Lab을 프로젝트 루트에서 열어 주세요.")


ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / CHAPTER_RELATIVE_DIR
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
sys.path.insert(0, str(ROOT / "common"))

from math_viz import PROFILES, run_profile

PROFILE = "computational_math"
TITLE = "계산수학 - Monte Carlo 방법"
CONCEPT_EXAMPLE = "무작위 표본으로 면적과 기대값을 추정한다."
APPLICATION_EXAMPLE = "고차원 적분을 확률적 샘플링으로 근사한다."

print("project root:", ROOT)
print("chapter dir:", NOTEBOOK_DIR)
print("profile:", PROFILE)

## 1. 이번 챕터의 시각화 코드 읽기

먼저 실제로 실행될 함수를 확인합니다. 코드를 읽으면서 입력값, 이산화 방식, 그래프가 의미하는 수학적 대상을 표시해 보세요.

In [ ]:
import inspect

print(inspect.getsource(PROFILES[PROFILE]))

## 2. 실행하고 결과 확인하기

아래 셀을 실행하면 `outputs/visualization.png`가 생성되고, 노트북 안에도 바로 표시됩니다.

In [ ]:
run_profile(
    profile=PROFILE,
    title=TITLE,
    concept=CONCEPT_EXAMPLE,
    application=APPLICATION_EXAMPLE,
    output_dir=OUTPUT_DIR,
)

display(Image(filename=str(OUTPUT_DIR / "visualization.png")))

## 3. 변형 실험

- 표본 수, 격자 크기, 초기값, 학습률, 경계조건 중 하나를 바꿔 보세요.
- 그림이 안정적으로 유지되는 범위와 결론이 바뀌는 범위를 나누어 적어 보세요.
- 손계산 가능한 작은 예제를 만들어 코드 결과와 비교해 보세요.

In [ ]:
# 여기에 자신만의 변형 실험을 작성하세요.
# 예: common/math_viz.py에서 위에 출력된 함수의 파라미터를 복사해 와서
#     표본 수, 구간, 초기값 등을 바꾼 뒤 다시 그려 볼 수 있습니다.
